In [1]:
!pip install datasets scikit-learn -q

In [2]:
from datasets import load_dataset
import torch

ds_test = load_dataset("dinushiTJ/nz_research_commons_inference_results", split="train")
display(ds_test)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse'],
    num_rows: 3318
})

In [3]:
print(ds_test.column_names)

['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse']


In [4]:
print(ds_test.features)

{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': ClassLabel(names=['maori_origin', 'non_maori_origin']), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'BaseEmbGemZeroLabel': Value('string'), 'BaseEmbGemZeroID': Value('int64'), 'BaseEmbGemZeroScore': Value('float64'), 'Base4BResponse': Value('string')}


In [5]:
display(ds_test[0]['Base4BResponse'])

'"```json\\n{\\n  \\"label\\": \\"maori_origin\\",\\n  \\"confidence\\": 0.9,\\n  \\"reason\\": \\"The research is focused on the experiences of female students transitioning to tertiary education in PNG. While not explicitly about Māori, the context of Papua New Guinea and the challenges faced by women pursuing education – particularly within a potentially patriarchal society – aligns with the criteria for ‘maori_origin’ due to the implicit connection to systemic barriers and the need for support and cultural change to facilitate educational success. The discussion of role models and family involvement is relevant to culturally-based support systems.\\"\\n}\\n```"'

In [6]:
import json

def extract_label_from_base4b_response(response_string):
    """
    Extracts the classification label from the Base4BResponse string.
    Assumes the label might be a JSON field or directly present in the text.
    """
    if not isinstance(response_string, str):
        return None

    # Strip the markdown code block delimiters if present
    cleaned_response_string = response_string.strip()
    if cleaned_response_string.startswith("```json") and cleaned_response_string.endswith("```"):
        cleaned_response_string = cleaned_response_string[len("```json"):].strip()
        cleaned_response_string = cleaned_response_string[:-len("```")].strip()

    # Try to parse as JSON first
    try:
        response_json = json.loads(cleaned_response_string) # Use the cleaned string here
        # Check if the parsed JSON is a dictionary before trying to access keys
        if isinstance(response_json, dict):
            if 'label' in response_json:
                return response_json['label']
        # If it's not a dict, but a simple string (e.g., '"maori_origin"'),
        # we can return it directly if it matches expected labels.
        elif isinstance(response_json, str):
            if response_json.lower() in ['maori_origin', 'non_maori_origin']:
                return response_json
    except json.JSONDecodeError:
        pass # Not a JSON string (even after cleaning), proceed to text search

    # If not JSON or key not found in JSON (or JSON was a simple string not matching expected labels),
    # search for known labels in the original text. Prioritize 'non_maori_origin' if both are present.
    if 'non_maori_origin' in response_string.lower():
        return 'non_maori_origin'
    elif 'maori_origin' in response_string.lower():
        return 'maori_origin'

    return None # No label found

Now, let's apply this function to the `Base4BResponse` column and see the extracted labels. We'll add a new column `extracted_label` to our dataset for this.

In [7]:
# Apply the function to the 'Base4BResponse' column
ds_test = ds_test.map(lambda example: {'Base4BLabel': extract_label_from_base4b_response(example['Base4BResponse'])})

# Display the original response and the extracted label for the first few examples
print("Original Base4BResponse and Extracted Label for the first 5 examples:")
for i in range(5):
    print(f"Example {i}:")
    print(f"  Base4BResponse: {ds_test[i]['Base4BResponse']}")
    print(f"  Base4BLabel: {ds_test[i]['Base4BLabel']}")
    print("-" * 30)

Map:   0%|          | 0/3318 [00:00<?, ? examples/s]

Original Base4BResponse and Extracted Label for the first 5 examples:
Example 0:
  Base4BResponse: "```json\n{\n  \"label\": \"maori_origin\",\n  \"confidence\": 0.9,\n  \"reason\": \"The research is focused on the experiences of female students transitioning to tertiary education in PNG. While not explicitly about Māori, the context of Papua New Guinea and the challenges faced by women pursuing education – particularly within a potentially patriarchal society – aligns with the criteria for ‘maori_origin’ due to the implicit connection to systemic barriers and the need for support and cultural change to facilitate educational success. The discussion of role models and family involvement is relevant to culturally-based support systems.\"\n}\n```"
  Base4BLabel: maori_origin
------------------------------
Example 1:
  Base4BResponse: "```json\n{\n  \"label\": \"non_maori_origin\",\n  \"confidence\": 0.75,\n  \"reason\": \"The research paper focuses on distributed operating systems and wi

In [11]:
none_count = 0
maori_count = 0
non_maori_count = 0
other_count = 0

for label in ds_test['Base4BLabel']:
    if label is None:
        none_count += 1
    elif label == 'maori_origin':
        maori_count += 1
    elif label == 'non_maori_origin':
        non_maori_count += 1
    else:
        other_count += 1

print(f"Count of 'maori_origin': {maori_count}")
print(f"Count of 'non_maori_origin': {non_maori_count}")
print(f"Count of None values: {none_count}")
print(f"Count of other (unrecognized) labels: {other_count}")

total_count = maori_count + non_maori_count + none_count + other_count
print(f"Total count: {total_count}")

if none_count > 0:
    print("\nThere are None values in the 'Base4BLabel' column, indicating that some labels could not be extracted.")
if other_count > 0:
    print("There are 'other' (unrecognized) labels, which might indicate an issue with label extraction or unexpected label values.")

Count of 'maori_origin': 856
Count of 'non_maori_origin': 2462
Count of None values: 0
Count of other (unrecognized) labels: 0
Total count: 3318


### Dataset Evaluation

Now, let's evaluate the dataset using the `classification_label` as the true labels and `BaseEmbGemZeroLabel` as the predicted labels.

In [12]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

# ClassLabel feature
class_label = ds_test.features["classification_label"]

# Ground truth IDs
true_labels_id = ds_test["classification_label"]

# Ground truth strings
true_labels = [
    class_label.int2str(label_id)
    for label_id in true_labels_id
]

# Predicted strings
pred_labels = ds_test["Base4BLabel"]

# Predicted IDs
predicted_labels_id = [
    class_label.str2int(label)
    for label in pred_labels
]

print(f"Unique true labels: {set(true_labels)}")
print(f"Unique true labels (IDs): {set(true_labels_id)}")

print(f"Unique predicted labels: {set(pred_labels)}")
print(f"Unique predicted labels (IDs): {set(predicted_labels_id)}")

Unique true labels: {'maori_origin', 'non_maori_origin'}
Unique true labels (IDs): {0, 1}
Unique predicted labels: {'maori_origin', 'non_maori_origin'}
Unique predicted labels (IDs): {0, 1}


In [13]:
label_names = class_label.names
label_to_id = {name: i for i, name in enumerate(label_names)}

In [14]:
# Calculate Confusion Matrix
conf_matrix = confusion_matrix(true_labels_id, predicted_labels_id, labels=[label_to_id[name] for name in label_names])
print("Confusion Matrix (task labels only):")
print(conf_matrix)

# Calculate Classification Report
class_report = classification_report(true_labels_id, predicted_labels_id, target_names=label_names, digits=4)
print("\nClassification Report (task labels only):")
print(class_report)

# Calculate Accuracy
accuracy = accuracy_score(true_labels_id, predicted_labels_id)
print(f"\nAccuracy: {accuracy:.4f}")

# Calculate F1 Score (macro and weighted)
f1_macro = f1_score(true_labels_id, predicted_labels_id, average='macro')
f1_weighted = f1_score(true_labels_id, predicted_labels_id, average='weighted')
print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Weighted: {f1_weighted:.4f}")

# Combine metrics into a dictionary as shown in the example
metrics = {
    'accuracy': accuracy,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted
}
print(f"\nCombined Metrics: {metrics}")

Confusion Matrix (task labels only):
[[ 358    9]
 [ 498 2453]]

Classification Report (task labels only):
                  precision    recall  f1-score   support

    maori_origin     0.4182    0.9755    0.5854       367
non_maori_origin     0.9963    0.8312    0.9063      2951

        accuracy                         0.8472      3318
       macro avg     0.7073    0.9034    0.7459      3318
    weighted avg     0.9324    0.8472    0.8708      3318


Accuracy: 0.8472
F1 Macro: 0.7459
F1 Weighted: 0.8708

Combined Metrics: {'accuracy': 0.8471971066907775, 'f1_macro': 0.7458911112960698, 'f1_weighted': 0.8708432316301629}


### Analyzing Misclassified Examples

Let's examine the rows where the `classification_label` does not match `Base4BLabel` to understand potential areas of misclassification.

In [17]:
# Find indices where true labels and predicted labels differ
mismatch_indices = [i for i, (true_id, pred_id) in enumerate(zip(true_labels_id, predicted_labels_id)) if true_id != pred_id]

# Get the mismatched examples from the dataset
mismatched_examples = ds_test.select(mismatch_indices)

# Display relevant columns for the first few mismatched examples
print(f"Found {len(mismatch_indices)} misclassified examples. Displaying the first 10:")

# Define the columns to display for better understanding
columns_to_display = ['title', 'abstract', 'classification_label', 'Base4BLabel', 'classification_reason']

# Convert ClassLabel IDs back to names for better readability in the display
mismatched_examples_df = mismatched_examples.to_pandas()
mismatched_examples_df['classification_label_name'] = mismatched_examples_df['classification_label'].map(lambda x: label_names[x])

# Select and display the relevant columns
display(mismatched_examples_df[columns_to_display + ['classification_label_name']].head(10))

Found 507 misclassified examples. Displaying the first 10:


,title,abstract,classification_label,Base4BLabel,classification_reason,classification_label_name
0,Exploring the experiences of female students' ...,Educational transitions take place when childr...,1,maori_origin,The research is focused exclusively on the edu...,non_maori_origin
1,"Knowledge, Experience and Attitudes towards sh...",Sharks are under threat from human exploitatio...,1,maori_origin,The research focuses on general public attitud...,non_maori_origin
2,Coastal Storm Activity along the Eastern North...,Coastal storm activity for the eastern North I...,1,maori_origin,The research is a purely physical science stud...,non_maori_origin
3,Language learning using Pacific picturebooks i...,Māori and Pacific learners in Aotearoa New Zea...,1,maori_origin,The research focuses specifically on the 'Paci...,non_maori_origin
4,Community and family literacies in Aotearoa Ne...,Video of Introduction in the Waikato Journal o...,1,maori_origin,The title and abstract indicate a general anal...,non_maori_origin
5,Redistribution and Recognition for Migrants an...,Located in demographically diverse Aotearoa Ne...,1,maori_origin,The research focuses exclusively on the settle...,non_maori_origin
6,The temperance movement in the Te Aroha district,As over-indulgence in strong drink was seen as...,1,maori_origin,The research focuses on the social and moral h...,non_maori_origin
7,A life in music: The contribution of John Ritc...,This thesis is a study of the life of New Zeal...,1,maori_origin,The research is a biographical study of John R...,non_maori_origin
8,Researching in cross cultural contexts: a soci...,"In this paper, we explore culture and its rela...",1,maori_origin,The research is explicitly focused on the Sout...,non_maori_origin
9,A decision support system for predicting the c...,Health information systems (HIS) serve as the ...,1,maori_origin,The research focuses on a general New Zealand ...,non_maori_origin
